In [ ]:
# With help from ChatGPT:

import time
import os
import zipfile
from selenium import webdriver

# Donwload ChromeDriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from webdriver_manager.chrome import ChromeDriverManager


options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# Automatically downloads the correct ChromeDriver and runs it
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


def wait_for_download(download_dir, timeout=200):
        seconds = 0
        dl_wait = True
        while dl_wait and seconds < timeout:
            time.sleep(1)
            files = [f for f in os.listdir(download_dir) if not f.endswith('.crdownload')]
            if files:
                # Get the most recent file
                latest_file = max([os.path.join(download_dir, f) for f in files], key=os.path.getctime)
    
                # Check if it's a ZIP file
                if latest_file.lower().endswith('.zip'):
                    dl_wait = False
            seconds += 1
            
        if not latest_file or not latest_file.lower().endswith('.zip'):
            raise Exception("Download failed, timed out, or the file is not a .zip archive.")

        return latest_file


# Extract and rename
def afterward(year, month, downloaded_file):
    print("Downloaded file:", downloaded_file)
    
    # Rename zip file
    old_path = downloaded_file
    new_name = str(year) + "-" + str(month)
    new_path = "/Users/ey3722/Downloads/" + new_name +".zip"
    #new_path = "/Users/erinyoo/Downloads/" + new_name +".zip"
    os.rename(old_path, new_path)
    print("Renamed to:", new_path)
    
    # Extract zip
    with zipfile.ZipFile(new_path, 'r') as zip_ref:
        zip_ref.extract('T_ONTIME_REPORTING.csv', download_dir)
        print('unzipped')
        os.rename('/Users/ey3722/Downloads/T_ONTIME_REPORTING.csv', '/Users/ey3722/Downloads/' + new_name + '.csv')
        #os.rename('/Users/erinyoo/Downloads/T_ONTIME_REPORTING.csv', '/Users/erinyoo/Downloads/' + new_name + '.csv')
        #time.sleep(10)

In [ ]:
# Go to BTS data
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

btnids = ["YEAR", "MONTH", "OP_UNIQUE_CARRIER", "ORIGIN_AIRPORT_SEQ_ID", "ORIGIN_CITY_MARKET_ID",
          "ORIGIN_CITY_NAME", "ORIGIN_STATE_ABR", "DEST_AIRPORT_SEQ_ID", "DEST_CITY_MARKET_ID",
          "DEST_CITY_NAME", "DEST_STATE_ABR", "DEP_DELAY", "ARR_DELAY", "CANCELLED", "CANCELLATION_CODE",
          "DIVERTED", "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"]

driver.get("https://www.transtats.bts.gov/Tables.asp?QO_VQ=EFD&QO_anzr=Nv4yv0r%FDb0-gvzr%FDcr4s14zn0pr%FDQn6n&QO_fu146_anzr=b0-gvzr")
driver.find_element(By.CSS_SELECTOR, 'a[href*="DL_SelectFields.asp?gnoyr_VQ=FGJ"]').click()

download_dir = '/Users/ey3722/Downloads'
#download_dir = '/Users/erinyoo/Downloads'

assert driver.title == "Download page", f"Unexpected title: {driver.title}"

# 1. Choose Year
for year in range(1998, 2024+1):

    dropdown_yr = Select(driver.find_element(By.ID, "cboYear"))
    dropdown_yr.select_by_value(str(year))
    dropdown_yr.first_selected_option
    
    # 2. Choose Month
    for month in range(12):
        
        # Clean up any partial or leftover files
        for f in os.listdir(download_dir):
            if f.endswith('.crdownload') or f.endswith('.zip'):
                try:
                    os.remove(os.path.join(download_dir, f))
                    print(f"Deleted leftover file: {f}")
                except Exception as e:
                    print(f"Could not delete {f}: {e}")
                
        dropdown_mth = Select(driver.find_element(By.ID, "cboPeriod"))
        dropdown_mth.select_by_index(month)
        selected_option = dropdown_mth.first_selected_option
    
        # 3. Reset year & fields to pull
        for btnid in btnids:
            driver.find_element(By.ID, btnid).click()

        dropdown_yr = Select(driver.find_element(By.ID, "cboYear"))
        dropdown_yr.select_by_value(str(year))
        dropdown_yr.first_selected_option


        # 4. Download
        #driver.find_element(By.ID, "btnDownload").click()
        checkbox = driver.find_element(By.ID, "btnDownload")
        driver.execute_script("arguments[0].click();", checkbox)
    
        # Optionally, print confirmation
        print("Download button clicked!")
        
        downloaded_file = wait_for_download(download_dir)
    
        afterward(year, month, downloaded_file)
        
        driver.refresh()  # Refresh the page to reset state

Download button clicked!
Downloaded file: /Users/ey3722/Downloads/T_ONTIME_REPORTING_20250501_221023.zip
Renamed to: /Users/ey3722/Downloads/1996-0.zip
unzipped
Deleted leftover file: 1996-0.zip
Download button clicked!
Downloaded file: /Users/ey3722/Downloads/T_ONTIME_REPORTING_20250501_221050.zip
Renamed to: /Users/ey3722/Downloads/1996-1.zip
unzipped
Deleted leftover file: 1996-1.zip
Download button clicked!
Downloaded file: /Users/ey3722/Downloads/T_ONTIME_REPORTING_20250501_221119.zip
Renamed to: /Users/ey3722/Downloads/1996-2.zip
unzipped
Deleted leftover file: 1996-2.zip
Download button clicked!
Downloaded file: /Users/ey3722/Downloads/T_ONTIME_REPORTING_20250501_221147.zip
Renamed to: /Users/ey3722/Downloads/1996-3.zip
unzipped
Deleted leftover file: 1996-3.zip
Download button clicked!
Downloaded file: /Users/ey3722/Downloads/T_ONTIME_REPORTING_20250501_221215.zip
Renamed to: /Users/ey3722/Downloads/1996-4.zip
unzipped
Deleted leftover file: 1996-4.zip
Download button clicked!
